# Data Integrity Compiler, Milestone 1 validation

Run this notebook on a Kaggle CPU runtime. The workload is DuckDB and does not need a GPU. Add `data-integrity-compiler-source.zip` as a notebook input, enable Internet, and run all cells. This notebook downloads official TLC files, checks their SHA-256 values, runs clean controls and the M1 fault experiment, independently reconciles the January Parquet outputs, and creates a downloadable results archive.

The report distinguishes pipeline execution from integrity outcome. A blocked fault run is expected. Build and upload the source ZIP together with its provenance sidecar. Record the Kaggle notebook version identifier in the provenance cell before running.

In [10]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, urllib.request, zipfile
from decimal import Decimal

inputs = Path("/kaggle/input")
archives = list(inputs.rglob("data-integrity-compiler-source.zip"))
ROOT = Path("/kaggle/working/data-integrity-compiler")
if ROOT.exists(): shutil.rmtree(ROOT)
if archives:
    ROOT.mkdir(parents=True)
    with zipfile.ZipFile(archives[0]) as z: z.extractall(ROOT)
else:
    srcs = [p.parent for p in inputs.rglob("SOURCE_REVISION.json") if (p.parent / "requirements-lock.txt").is_file()]
    if not srcs: raise FileNotFoundError("Extracted project (with SOURCE_REVISION.json) not found under /kaggle/input")
    shutil.copytree(srcs[0], ROOT)
print("Project ready at", ROOT)
print("Python", sys.version)

Project ready at /kaggle/working/data-integrity-compiler
Python 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [11]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv'], check=True)
VENV = ROOT / '.venv'
shutil.rmtree(VENV, ignore_errors=True)   # remove the half-built venv from the failed attempt
subprocess.run([sys.executable, '-m', 'virtualenv', '-q', str(VENV)], check=True)
PYTHON = str(VENV / 'bin' / 'python')
subprocess.run([PYTHON, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements-lock.txt')], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(ROOT)], check=True)
print("Pinned project dependencies installed")

Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


Pinned project dependencies installed


In [12]:
raw = ROOT / "data" / "raw"
raw.mkdir(parents=True, exist_ok=True)
expected = {
    "yellow_tripdata_2024-01.parquet": "c4d59da7bbc8abaeeeb1727947ee93d9891a71acb42854bd80db1571b2030510",
    "yellow_tripdata_2024-02.parquet": "c76c43c18c6c6664080dd920baab4928988d5786a6b65980792ca7cd796f9f20",
    "yellow_tripdata_2025-01.parquet": "9af277e4c0d3f9deb30644da822981e1e7df6af58313170fd3aa8a474485488a",
    "taxi_zone_lookup.csv": "1a99e105092230f8620f301edcca7f80d3080642ff404d28ed957d3fa222c8ed",
}
urls = {name: ("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv" if name == "taxi_zone_lookup.csv" else "https://d37ci6vzurychx.cloudfront.net/trip-data/" + name) for name in expected}
def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()
sidecars = list(inputs.rglob('data-integrity-compiler-source.provenance.json'))
if not sidecars: raise FileNotFoundError('Provenance sidecar not found in the dataset')
source_provenance = json.loads(sidecars[0].read_text())
if archives:
    assert source_provenance['source_archive_sha256'] == sha256(archives[0]), 'Source archive checksum differs from its provenance sidecar'
else:
    print('NOTE: Kaggle extracted the ZIP; archive hash will be verified locally by package_m1_evidence.py')
embedded_source = json.loads((ROOT/'SOURCE_REVISION.json').read_text())
assert embedded_source['commit_sha'] == source_provenance['commit_sha'] and embedded_source['tree_sha'] == source_provenance['tree_sha']
KAGGLE_NOTEBOOK_VERSION = "m1-1-run-1"
if not KAGGLE_NOTEBOOK_VERSION: raise ValueError('Set KAGGLE_NOTEBOOK_VERSION to the saved Kaggle version identifier before the full run')
for name,url in urls.items():
    target=raw/name
    if not target.exists(): urllib.request.urlretrieve(url,target)
    actual=sha256(target)
    assert actual == expected[name], f"Checksum mismatch for {name}: {actual}"
print("All official source checksums verified")

NOTE: Kaggle extracted the ZIP; archive hash will be verified locally by package_m1_evidence.py
All official source checksums verified


In [13]:
def dic(*args):
    result=subprocess.run([PYTHON,"-m","dic.cli",*map(str,args)],cwd=ROOT,text=True,capture_output=True)
    if result.returncode: raise RuntimeError(result.stderr[-4000:] or result.stdout[-4000:])
    return json.loads(result.stdout)
for month in ("2024-01","2024-02","2025-01"):
    meta=dic("acquire","--month",month)
    print(month, meta["schema_classification"], meta["added_columns"])
    assert meta["schema_classification"] == "compatible"
manifest=json.loads((ROOT/"data/source-manifest.json").read_text())
assert manifest["months"]["2025-01"]["added_columns"] == ["cbd_congestion_fee"]
assert manifest["months"]["2024-02"]["added_columns"] == []

2024-01 compatible []
2024-02 compatible []
2025-01 compatible ['cbd_congestion_fee']


In [14]:
controls=ROOT/"data/clean-controls"
controls.mkdir(parents=True,exist_ok=True)
reports={}
for month in ("2024-02","2025-01","2024-01","2024-01"):
    report=dic("run","--month",month)
    assert report["outcome"] == "passed", month
    reports.setdefault(month,[]).append(report)
    suffix="-repeat" if len(reports[month]) == 2 else ""
    (controls/f"{month}{suffix}.json").write_text(json.dumps(report,indent=2),encoding="utf-8")
    print(month, report["run_id"], report["runtimes_seconds"])
jan_clean=reports["2024-01"][0]
jan_repeat=reports["2024-01"][1]
assert jan_clean["logical_output_hashes"] == jan_repeat["logical_output_hashes"]
assert all(all(c["passed"] for c in r["checks"]) for rs in reports.values() for r in rs)
print("Clean controls passed; January logical fingerprints repeat")

2024-02 62775395f3a24407adea2b47464d4f2a {'total_pipeline_seconds': 7.268712, 'validation_seconds': 0.581011, 'output_materialization_seconds': 4.570472, 'publication_seconds': 0.025822}
2025-01 bc83f61d30d44ce68854a32e5697ef66 {'total_pipeline_seconds': 8.136514, 'validation_seconds': 0.635856, 'output_materialization_seconds': 5.308246, 'publication_seconds': 0.030903}
2024-01 80c94a2eb9384d2a948fb64901ada60a {'total_pipeline_seconds': 7.47238, 'validation_seconds': 0.583536, 'output_materialization_seconds': 4.82861, 'publication_seconds': 0.025758}
2024-01 873ed28085ed4b5a91009b9dc6f8797a {'total_pipeline_seconds': 7.06865, 'validation_seconds': 0.566071, 'output_materialization_seconds': 4.581431, 'publication_seconds': 0.025175}
Clean controls passed; January logical fingerprints repeat


In [15]:
injected=dic("inject-m1","--month","2024-01","--key","1")
fault=dic("run","--month","2024-01","--lookup",injected["lookup"],"--reference-lookup",raw/"taxi_zone_lookup.csv","--fault-id",injected["fault_id"])
assert fault["outcome"] == "blocked"
(ROOT/"data/fault_report.json").write_text(json.dumps(fault,indent=2),encoding="utf-8")
(ROOT/"data/clean_report.json").write_text(json.dumps(jan_clean,indent=2),encoding="utf-8")
evaluated=dic("evaluate","--report",ROOT/"data/runs"/fault["run_id"]/"report.json","--ground-truth",injected["ground_truth"])
print("Fault evaluation:",evaluated)
assert evaluated["fault_id_matches"] and evaluated["key_matches"] and evaluated["invariant_detected"]
assert fault["publication"]["pointer_before"] == fault["publication"]["pointer_after"] == jan_repeat["run_id"]
print("Fault blocked and prior clean publication pointer was preserved")

Fault evaluation: {'fault_id_matches': True, 'key_matches': True, 'invariant_detected': True}
Fault blocked and prior clean publication pointer was preserved


In [16]:
# Independent comparison reads the saved Parquet outputs, not the detector ground truth.
subprocess.run([PYTHON,str(ROOT/"experiments/reconcile_reports.py"),"--root",str(ROOT)],cwd=ROOT,check=True)
reconciliation=json.loads((ROOT/"data/jan-2024-reconciliation.json").read_text())
assert reconciliation["expected_row_inflation"] == reconciliation["observed_row_inflation"]
assert reconciliation["report_aggregate_deltas_match_parquet"]
assert reconciliation["average_fares_unchanged_in_all_affected_groups"]
assert fault["expected_observed"]["row_inflation"] == reconciliation["observed_row_inflation"]
assert reconciliation["clean_run_id"] == jan_clean["run_id"]
assert reconciliation["contract_sha256"] == jan_clean["contract_hash"]
assert all(sha256(raw/name)==expected[name] for name in expected)
print(json.dumps(reconciliation,indent=2))

{
  "reconciliation_version": "jan-2024-parquet-reconciliation-v1",
  "clean_run_id": "80c94a2eb9384d2a948fb64901ada60a",
  "fault_run_id": "5f63dbd47d0d417db0848436d5c1b791",
  "clean_manifest": "data/runs/80c94a2eb9384d2a948fb64901ada60a/manifest.json",
  "fault_manifest": "data/runs/5f63dbd47d0d417db0848436d5c1b791/manifest.json",
  "contract_sha256": "947e696d07ed07482a9d542febf5a345693e5a1a29b4db5875dc0fbdeca3c953",
  "clean_diagnostic_rows": 2870188,
  "fault_diagnostic_rows": 2870238,
  "matching_input_rows_for_M1": 50,
  "expected_row_inflation": 50,
  "observed_row_inflation": 50,
  "affected_daily_zone_groups": 22,
  "total_trip_count_delta": 50,
  "total_revenue_delta": "4134.35",
  "average_fares_unchanged_in_all_affected_groups": true,
  "report_aggregate_deltas_match_parquet": true,
  "publication_pointer_preserved": true,
  "limitations": [
    "One injected join fault is a plumbing and attribution test, not a recall or false-positive estimate.",
    "Peak memory is meas

Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


{
  "reconciliation_version": "jan-2024-parquet-reconciliation-v1",
  "clean_run_id": "80c94a2eb9384d2a948fb64901ada60a",
  "fault_run_id": "5f63dbd47d0d417db0848436d5c1b791",
  "clean_manifest": "data/runs/80c94a2eb9384d2a948fb64901ada60a/manifest.json",
  "fault_manifest": "data/runs/5f63dbd47d0d417db0848436d5c1b791/manifest.json",
  "contract_sha256": "947e696d07ed07482a9d542febf5a345693e5a1a29b4db5875dc0fbdeca3c953",
  "clean_diagnostic_rows": 2870188,
  "fault_diagnostic_rows": 2870238,
  "matching_input_rows_for_M1": 50,
  "expected_row_inflation": 50,
  "observed_row_inflation": 50,
  "affected_daily_zone_groups": 22,
  "total_trip_count_delta": 50,
  "total_revenue_delta": "4134.35",
  "average_fares_unchanged_in_all_affected_groups": true,
  "report_aggregate_deltas_match_parquet": true,
  "publication_pointer_preserved": true,
  "limitations": [
    "One injected join fault is a plumbing and attribution test, not a recall or false-positive estimate.",
    "Peak memory is meas

In [17]:
# Export reports and compact manifests; keep raw data and private ground truth outside this bundle.
export=Path("/kaggle/working/dic-m1-validation-results.zip")
provenance={**source_provenance,"kaggle_notebook_version":KAGGLE_NOTEBOOK_VERSION,"notebook_sha256":sha256(ROOT/"experiments/kaggle_upload/kaggle_m1_validation.ipynb")}
(ROOT/"data/provenance.json").write_text(json.dumps(provenance,indent=2),encoding="utf-8")
keep=[ROOT/"data/source-manifest.json",ROOT/"data/jan-2024-reconciliation.json",ROOT/"data/clean_report.json",ROOT/"data/fault_report.json",ROOT/"data/provenance.json",*controls.glob("*.json"),ROOT/"contracts/pipeline.yaml",ROOT/"contracts/tlc-yellow-v1.json",ROOT/"schemas/report-v2.schema.json"]
# Include the exact clean run referenced by reconciliation, its manifest, the repeat run, and the fault run.
for run_id in (jan_clean["run_id"],jan_repeat["run_id"],fault["run_id"]):
    d=ROOT/"data/runs"/run_id
    keep.extend(p for p in d.iterdir() if p.is_file() and p.suffix in (".parquet",".json",".csv"))
clean_manifest=json.loads((ROOT/"data/runs"/jan_clean["run_id"]/"manifest.json").read_text())
assert clean_manifest["contract_sha256"] == reconciliation["contract_sha256"]
published=ROOT/"data/published"
if (published/"CURRENT").exists(): keep.append(published/"CURRENT")
with zipfile.ZipFile(export,"w",zipfile.ZIP_DEFLATED,compresslevel=3) as z:
    for p in keep:
        if p.exists(): z.write(p,p.relative_to(ROOT).as_posix())
print("Download this output:",export)
print("Validation complete")

Download this output: /kaggle/working/dic-m1-validation-results.zip
Validation complete
